# LeetCode #1140 — Stone Game II: Trò chơi lấy đá II

**Medium · Python 3 · Game Theory (Minimax) · Dynamic Programming · Memoization**  
[Đề gốc](https://leetcode.com/problems/stone-game-ii/) · [Đối chiếu mô tả](https://leetcode.doocs.org/en/lc/1140/)

```python
class Solution:
    def stoneGameII(self, piles: List[int]) -> int:
        ...
```

**Câu hỏi xuyên suốt:** Khi ta lấy nhiều đá ở lượt này, có thể vô tình cho đối thủ một nước rất mạnh ở lượt sau. Làm sao chọn **tối ưu trước một đối thủ cũng tối ưu**, thay vì chỉ tham ở lượt hiện tại?

## 1. Sau bài này, bạn sẽ tự làm được gì?

1. Mô phỏng luật chơi, nhận ra trạng thái gồm **vị trí `i` và giới hạn `M`**.
2. Hiểu `minimax`: tối đa hóa phần mình, nhưng đối thủ cũng tối đa phần của họ.
3. Viết lời giải vét cạn trước rồi nhận ra các trạng thái bị lặp.
4. Dùng `suffix sum` và `@cache` để giảm tính lại.
5. Theo dõi từng lựa chọn `X` bằng trace, chứng minh công thức.
6. Viết `class Solution`, kiểm thử và hoàn thiện 5 bài tập TODO.

## 2. Đề bài, giải nghĩa chính xác và ví dụ

Có các đống đá xếp theo hàng, `piles[i]` là số viên đá ở đống thứ `i` (**đều dương**). Alice đi trước, Bob đi sau. Đầu ván `M=1`.

Mỗi lượt, người chơi chọn một số nguyên `X` thỏa **`1 <= X <= 2*M`**, lấy **toàn bộ `X` đống đầu tiên còn lại**, rồi cập nhật `M = max(M, X)`.

Trò chơi hết khi mọi đống đã bị lấy. Cả hai chơi tối ưu. **Trả về số viên đá lớn nhất Alice có thể lấy được** (không trả tổng hai người, không trả hiệu số).

| Input | Đáp án | Ý nghĩa |
|---|---:|---|
| `[2,7,9,4,4]` | **10** | Alice lấy `2`, Bob lấy `7+9`, Alice lấy `4+4` |
| `[1,2,3,4,5,100]` | **104** | Phải tính cả lợi ích về sau, không chỉ chọn đống lớn ngay |

**Constraints:** `1 <= len(piles) <= 100`, `1 <= piles[i] <= 10^4`.

In [1]:
piles_demo = [2, 7, 9, 4, 4]
print('Các đống đá:', piles_demo)
print('Tổng số viên:', sum(piles_demo))
print('Lượt mở đầu: X có thể là 1 hoặc 2 (M=1).')
assert sum(piles_demo) == 26

Các đống đá: [2, 7, 9, 4, 4]
Tổng số viên: 26
Lượt mở đầu: X có thể là 1 hoặc 2 (M=1).


## 3. Python nền tảng: index, slicing, sum và range

- `piles[i]`: giá trị tại chỉ số `i` (**bắt đầu từ 0**).
- `piles[i:j]`: lấy đoạn từ chỉ số `i` đến `j-1` (không lấy `j`).
- `sum(piles[i:i+X])`: số đá lấy nếu người chơi đang đứng ở `i` và chọn `X` đống.
- `range(1, limit+1)`: lần lượt xét các giá trị `1, 2, ..., limit`.

Ta luôn chỉ lấy **đoạn liên tiếp ở đầu phần chưa lấy**, không được bỏ qua đống giữa.

In [2]:
sample = [2, 7, 9, 4, 4]
for X in range(1, 3):
    print(f'X={X}: lấy {sample[:X]}, được {sum(sample[:X])} viên')
assert sample[1:3] == [7, 9]

X=1: lấy [2], được 2 viên
X=2: lấy [2, 7], được 9 viên


## 4. Bảng trạng thái khi thay lượt

**`i`** = vị trí đống đầu tiên chưa lấy. **`M`** = giới hạn của lượt hiện tại. Nếu chọn `X`:

- Lấy đoạn `piles[i:i+X]`.
- Lượt sau bắt đầu ở `i' = i+X`.
- Lượt sau có `M' = max(M, X)`.

Tại `(i=0, M=1)`, Alice chọn `X=1` thì Bob ở `(1,1)`; chọn `X=2` thì Bob ở `(2,2)`. Bob sẽ lựa chọn để **tối đa phần Bob**, không giúp Alice.

In [3]:
for X in (1, 2):
    print('Alice chọn X=', X,
          '| đá Alice vừa lấy=', sum(piles_demo[:X]),
          '| trạng thái Bob=', (X, max(1, X)))

Alice chọn X= 1 | đá Alice vừa lấy= 2 | trạng thái Bob= (1, 1)
Alice chọn X= 2 | đá Alice vừa lấy= 9 | trạng thái Bob= (2, 2)


## 5. Cách trực giác: thử mọi nước đi của cả hai

Ở một trạng thái `(i,M)`, chọn thử **tất cả** `X` hợp lệ và giả sử đối thủ cũng làm tương tự. Ta cần tìm phương án tốt nhất sau toàn bộ những lựa chọn đó.

Cách này đúng về mặt ý tưởng nhưng một trạng thái `(i,M)` có thể xuất hiện lại từ nhiều dãy nước đi. Nếu cứ gọi đệ quy lại từ đầu, số nhánh tăng rất nhanh.

Trước tiên hãy tách phép tính thành hàm nhận **người đang chơi**, không phải chỉ Alice. Sự đối xứng là điểm mấu chốt.

In [4]:
def brute_stone_game(piles):
    """Vét cạn không nhớ kết quả: chỉ dùng trên mảng ngắn."""
    n = len(piles)
    def play(i, M):
        if i == n:
            return 0
        best = 0
        for X in range(1, min(2*M, n-i)+1):
            gained = sum(piles[i:i+X])
            # Đá về sau = đá còn lại - phần tối ưu của đối thủ.
            remaining_after = sum(piles[i+X:])
            own_total = gained + remaining_after - play(i+X, max(M, X))
            best = max(best, own_total)
        return best
    return play(0, 1)

for a in ([1], [1, 2], [2, 1, 3], [2, 7, 9, 4, 4]):
    print(a, '→', brute_stone_game(a))
assert brute_stone_game([2, 7, 9, 4, 4]) == 10

[1] → 1
[1, 2] → 3
[2, 1, 3] → 3
[2, 7, 9, 4, 4] → 10


## 6. Vì sao là «tổng còn lại − phần đối thủ»?

Giả sử từ vị trí `i` trở đi có **17 viên**. Sau khi ta chọn `X`, người đối thủ sẽ lấy tối ưu được **12 viên** trong phần còn lại. Vì **mọi viên đá cuối cùng đều thuộc về một trong hai người**, số đá **của ta tính từ trạng thái ban đầu này** sẽ là `17 - 12 = 5`.

Đó là tư duy **zero-sum**: hai phần đá cộng lại đúng bằng tổng đá còn lại. Ta **không cần** lưu lượt Alice/Bob trong trạng thái, vì `play(i,M)` luôn nghĩa là **người sắp đi** có thể lấy tối đa bao nhiêu.

Công thức này đã gộp số viên lấy ngay lượt này với số viên lấy về sau.

In [5]:
total_remaining = 17
opponent_best = 12
my_best_if_choose_this_X = total_remaining - opponent_best
print('Phần mình có thể giữ:', my_best_if_choose_this_X)
assert my_best_if_choose_this_X == 5

Phần mình có thể giữ: 5


## 7. Dynamic Programming: xác định đúng trạng thái

Đặt `dp(i, M)` = **số đá tối đa của người sắp đi**, khi các đống `piles[i:]` còn lại và giới hạn hiện tại là `M`.

Phải nhớ **cả `i` lẫn `M`**:

- `i` khác nhau ⇒ số đá còn lại khác nhau.
- cùng `i` nhưng `M` khác nhau ⇒ số đống có thể lấy khác nhau.

Không có biến `turn`: cả hai người đều chơi theo đúng một quy tắc tối ưu. Đáp án: `dp(0, 1)`.

## 8. Suffix sum — giảm phép cộng lặp

`suffix[i]` là tổng đá từ `piles[i]` đến cuối. Tạo bảng từ **phải sang trái**:

\[suffix[i] = piles[i] + suffix[i+1],\qquad suffix[n]=0.\]

Với `[2,7,9,4,4]`, bảng sẽ là `[26,24,17,8,4,0]`.

Sau đó `suffix[i]` lấy được trong `O(1)`, thay vì gọi `sum(piles[i:])` mỗi nhánh.

In [6]:
def build_suffix(piles):
    n = len(piles)
    suffix = [0] * (n+1)
    for i in range(n-1, -1, -1):
        suffix[i] = piles[i] + suffix[i+1]
    return suffix

suffix_demo = build_suffix(piles_demo)
print('piles :', piles_demo)
print('suffix:', suffix_demo)
assert suffix_demo == [26, 24, 17, 8, 4, 0]

piles : [2, 7, 9, 4, 4]
suffix: [26, 24, 17, 8, 4, 0]


## 9. Nhắc lại Python: vòng lặp lùi và mảng `n+1`

`range(n-1, -1, -1)` duyệt `n-1, n-2, ..., 0`. Nếu quên bước `-1`, vòng lặp sẽ không chạy như ý.

`suffix` có **n+1 ô** để `suffix[n]=0`: khi đã lấy hết, không còn đá. Giá trị `suffix[i+1]` luôn tồn tại ngay cả tại `i=n-1`.

In [7]:
print('Các chỉ số duyệt ngược:', list(range(4, -1, -1)))
print('Tổng 3 đống cuối = suffix[2]:', suffix_demo[2])
assert suffix_demo[5] == 0 and suffix_demo[2] == 17

Các chỉ số duyệt ngược: [4, 3, 2, 1, 0]
Tổng 3 đống cuối = suffix[2]: 17


## 10. Công thức Minimax — tự suy ra

Khi `2*M >= n-i`, người đang chơi **được phép lấy hết**, nên `dp(i,M)=suffix[i]` (các đống dương; lấy hết là tối ưu).

Ngược lại, với `X` từ `1` đến `2*M`:

\[dp(i,M)=\max_{1\le X\le 2M}\Big(suffix[i] - dp(i+X,\max(M,X))\Big).\]

Ở đây `dp(i+X, ...)` là lượng **đối thủ** có thể lấy tối đa sau nước `X` của ta. Ta chọn `X` sao cho **đối thủ được ít nhất**, tương đương **mình được nhiều nhất**.

Đừng nhầm công thức thành `suffix[i] + dp(...)` — như thế sẽ đếm trùng viên đá.

## 11. Memoization là gì? `@cache` để làm gì?

**Memoization** = nhớ đáp án của một hàm theo tham số. `@cache` (Python `functools`) tự lưu kết quả `dp(i,M)` ở lần đầu. Khi hàm được gọi lại với **đúng cặp tham số**, Python trả kết quả đã lưu.

Đây chính là **top-down DP**: bắt đầu từ câu hỏi lớn, gọi xuống những câu hỏi nhỏ; bộ nhớ giúp tránh lời giải lặp.

In [8]:
from functools import cache

def demo_cache():
    times = {'computed': 0}
    @cache
    def square(x):
        times['computed'] += 1
        return x*x
    print(square(4), square(4), square(5))
    print('Số lần tính mới:', times['computed'])
    return times['computed']

assert demo_cache() == 2

16 16 25
Số lần tính mới: 2


## 12. Lời giải chính: `suffix + @cache`

Đọc theo thứ tự:

1. Tính `suffix` trước.
2. Trong `dfs(i,M)`, nếu được lấy hết thì trả `suffix[i]`.
3. Nếu không, thử `X` từ `1` đến `2*M`.
4. Dùng `suffix[i]-dfs(next_i,next_M)` để tính số đá **của người hiện tại**.
5. Chọn kết quả lớn nhất và trả `dfs(0,1)`.

In [9]:
def stone_game_memo(piles):
    n = len(piles)
    suffix = build_suffix(piles)

    @cache
    def dfs(i, M):
        if 2*M >= n-i:
            return suffix[i]
        best = 0
        for X in range(1, 2*M+1):
            opponent = dfs(i+X, max(M, X))
            best = max(best, suffix[i]-opponent)
        return best

    return dfs(0,1)

assert stone_game_memo([2,7,9,4,4]) == 10
assert stone_game_memo([1,2,3,4,5,100]) == 104
print('Mẫu 1:', stone_game_memo([2,7,9,4,4]))
print('Mẫu 2:', stone_game_memo([1,2,3,4,5,100]))

Mẫu 1: 10
Mẫu 2: 104


## 13. TRACE bằng tay trên `[2,7,9,4,4]`

Trạng thái bắt đầu `(i=0,M=1)`, `suffix[0]=26`. Alice chỉ có `X=1` hoặc `X=2`.

| Alice lấy X | Bob bắt đầu | Bob tối ưu | Phần Alice có thể giữ | 
|---:|---|---:|---:|
| 1 | `(1,1)` | 16 | `26-16=10` |
| 2 | `(2,2)` | 17 | `26-17=9` |

Alice chọn `X=1` để đạt **10**. Điểm quan trọng: *lấy ít hơn lúc đầu* lại có thể thắng về tổng cuối.

In [10]:
def trace_stone_game(piles, show_max=35):
    n, suffix = len(piles), build_suffix(piles)
    shown = [0]
    @cache
    def dfs(i,M):
        if 2*M >= n-i:
            return suffix[i]
        best = 0
        for X in range(1,2*M+1):
            opp = dfs(i+X,max(M,X))
            mine = suffix[i]-opp
            if shown[0] < show_max:
                print(f'i={i}, M={M}, X={X}: còn={suffix[i]}, '
                      f'đối thủ={opp} -> mình={mine}')
                shown[0] += 1
            best=max(best,mine)
        return best
    answer=dfs(0,1)
    print('ĐÁP ÁN:', answer, '| thông tin cache:', dfs.cache_info())
    return answer

assert trace_stone_game([2,7,9,4,4]) == 10

i=2, M=1, X=1: còn=17, đối thủ=8 -> mình=9
i=2, M=1, X=2: còn=17, đối thủ=4 -> mình=13
i=1, M=1, X=1: còn=24, đối thủ=13 -> mình=11
i=1, M=1, X=2: còn=24, đối thủ=8 -> mình=16
i=0, M=1, X=1: còn=26, đối thủ=16 -> mình=10
i=0, M=1, X=2: còn=26, đối thủ=17 -> mình=9
ĐÁP ÁN: 10 | thông tin cache: CacheInfo(hits=0, misses=7, maxsize=None, currsize=7)


## 14. Trace có thể in theo thứ tự «tính xong nhánh»

Vì DFS đi sâu trước, các dòng trace có thể **không** hiện `i=0` đầu tiên: chương trình phải tính nhánh con rồi mới biết Bob tối ưu được bao nhiêu. Hãy tìm hai dòng có `i=0` và so kết quả `10` với `9`.

Nếu muốn trace thứ tự từ trạng thái đầu, một cách là **in lúc bắt đầu gọi hàm**; nhưng kết quả `mine` chỉ in được *sau khi* lời gọi cho đối thủ hoàn tất.

In [11]:
p = [2,7,9,4,4]
print('Tại i=0:')
for X in (1,2):
    # Không tính lại trên p[X:] vì như thế sẽ đặt lại M=1 (SAI trạng thái).
    print('Chọn X=',X,'-> chuyển tới trạng thái', (X,max(1,X)))
print('Bắt buộc giữ nguyên cả i và M khi gọi hàm con!')

Tại i=0:
Chọn X= 1 -> chuyển tới trạng thái (1, 1)
Chọn X= 2 -> chuyển tới trạng thái (2, 2)
Bắt buộc giữ nguyên cả i và M khi gọi hàm con!


## 15. Code nộp LeetCode (tự chứa, đúng signature)

Trên LeetCode, ta gửi một `class Solution` duy nhất. Từ bản demo, ta thay tên hàm ngoài thành method `stoneGameII`, dùng `self` ở vị trí đầu. Không cần `print` và không để test trong phần submit.

In [12]:
from functools import cache
from typing import List

class Solution:
    def stoneGameII(self, piles: List[int]) -> int:
        n = len(piles)
        suffix = [0]*(n+1)
        for i in range(n-1,-1,-1):
            suffix[i] = piles[i] + suffix[i+1]

        @cache
        def dfs(i: int, M: int) -> int:
            if 2*M >= n-i:
                return suffix[i]
            best = 0
            for X in range(1,2*M+1):
                opponent = dfs(i+X, max(M,X))
                best = max(best, suffix[i]-opponent)
            return best

        return dfs(0,1)

solver = Solution()
assert solver.stoneGameII([2,7,9,4,4]) == 10
assert solver.stoneGameII([1,2,3,4,5,100]) == 104
print('LeetCode Solution: PASS')

LeetCode Solution: PASS


## 16. Vì sao thuật toán đúng?

**Quy nạp theo số đống còn lại.**

- Nếu `2M >= n-i`, người chơi hiện tại có thể lấy hết, đó là kết quả tốt nhất vì mọi đống đều dương.
- Nếu không, bất kỳ nước đi hợp lệ nào cũng tương ứng một `X` trong vòng lặp. Sau nước này, đối thủ nhận tối đa `dfs(i+X,max(M,X))` theo giả thiết quy nạp. Phần người hiện tại giữ được cuối ván bằng tổng còn lại trừ phần đối thủ. Lấy `max` trên mọi `X` chọn nước tối ưu.

Vì mọi trạng thái nhỏ hơn được giải đúng, `dfs(0,1)` cho phần Alice tối ưu. `@cache` chỉ bỏ tính lặp chứ không đổi kết quả.

## 17. Time / Space Complexity — giải thích cẩn thận

Gọi `n=len(piles)`.

- Có tối đa **O(n²)** trạng thái khác nhau `(i,M)` có thể xảy ra (với `M` không vượt số đống đã lấy, nên không tăng vô hạn).
- Mỗi trạng thái thử tối đa `2M <= O(n)` phương án ⇒ **O(n³)** thời gian *cận trên*.
- `suffix` chiếm `O(n)` và cache nhớ tối đa `O(n²)` kết quả; stack đệ quy sâu tối đa `O(n)` ⇒ **O(n²)** bộ nhớ phụ tổng.
- Vét cạn không cache có số nhánh tăng theo cấp số nhân trong trường hợp xấu.

`@cache` dùng không gian đổi lấy thời gian; với `n <= 100` độ sâu đệ quy là an toàn trong giới hạn Python thông thường.

## 18. Kiểm thử ví dụ, biên, trường hợp dễ nhầm

- Một đống: Alice lấy hết.
- Hai đống: Alice được phép lấy cả hai ngay từ lượt đầu.
- Đống cuối rất lớn: chiến lược ngây thơ dễ thất bại.
- Đống bằng nhau: không được ngầm giả định chỉ có một phương án tối ưu.

In [13]:
cases = [
    ([1],1),
    ([10000],10000),
    ([1,2],3),
    ([7,7],14),
    ([2,7,9,4,4],10),
    ([1,2,3,4,5,100],104),
]
for arr,expected in cases:
    result=Solution().stoneGameII(arr)
    assert result==expected,(arr,result,expected)
    print('PASS',arr,'->',result)

PASS [1] -> 1
PASS [10000] -> 10000
PASS [1, 2] -> 3
PASS [7, 7] -> 14
PASS [2, 7, 9, 4, 4] -> 10
PASS [1, 2, 3, 4, 5, 100] -> 104


## 19. Test đối chiếu ngẫu nhiên với vét cạn (oracle)

**Oracle** là cách giải độc lập chậm nhưng dễ tin trên input nhỏ. Ta phát sinh mảng ngắn, so kết quả DP với brute force. Đây là cách tìm lỗi trạng thái rất hữu ích: sai `M'`, quên chuyển lượt, hay trừ nhầm phần đối thủ đều dễ lộ ra.

In [14]:
import random
rng = random.Random(1140)
for _ in range(170):
    arr=[rng.randint(1,15) for _ in range(rng.randint(1,7))]
    expected=brute_stone_game(arr)
    actual=Solution().stoneGameII(arr)
    assert actual==expected,(arr,actual,expected)
print('PASS 170 ca đối chiếu ngẫu nhiên')

PASS 170 ca đối chiếu ngẫu nhiên


## 20. Kiểm thử cận lớn của đề

`n=100` là trường hợp mà đệ quy vét cạn rất chậm. Cache chỉ giữ lại mỗi trạng thái khác nhau một lần. Không cần đo thời gian cụ thể vì phụ thuộc máy chạy.

In [15]:
arr_big = [((i*7919) % 10000)+1 for i in range(100)]
answer_big = Solution().stoneGameII(arr_big)
assert 0 < answer_big <= sum(arr_big)
assert answer_big == stone_game_memo(arr_big)
print('PASS n=100; Alice có:', answer_big, 'trong tổng', sum(arr_big))

PASS n=100; Alice có: 246544 trong tổng 489150


## 21. Lỗi thường gặp và cách sửa

| Sai | Vì sao sai | Sửa |
|---|---|---|
| Chỉ chọn `X` lớn nhất | Lượt sau Bob cũng được lợi khi `M` lớn | Duyệt **mọi** `1..2M` |
| Cập nhật `M += X` | Luật gốc là lấy giá trị lớn nhất | `max(M,X)` |
| Lưu `dp(i)` | Cùng vị trí nhưng giới hạn số đống khác | Lưu `dp(i,M)` |
| `suffix[i] + dp(...)` | Đếm trùng đá | `suffix[i] - dp(...)` |
| Gọi DP mới trên `piles[i:]` mỗi bước | Đặt lại `M=1`, làm sai trạng thái | Giữ nguyên `i,M` trong cùng lời gọi |
| Lặp `range(1,2*M)` | Bỏ sót `X=2M` | `range(1,2*M+1)` |

## 22. Khi nào nhận ra pattern Minimax + Memoization?

Dấu hiệu: hai người **lần lượt** ra quyết định; cả hai đều tối ưu; câu trả lời phụ thuộc vào **trạng thái còn lại**, không chỉ lựa chọn hiện tại.

Công thức thường có dạng `best(state) = max(số-lợi-đang-có - best(next_state))` trong trò chơi tổng không đổi. Nếu trò chơi tính **thắng/thua** thay vì tổng điểm, trạng thái có thể lưu `True/False`; nếu tính **hiệu điểm**, ta dùng `max(gained - best_difference(next))`. Không thay các công thức này cho nhau mà không xác định ý nghĩa trạng thái.

## 23. Năm bài tập (làm trước khi mở gợi ý)

1. **Tính tay:** Với `piles=[1,2]`, Alice có những lựa chọn `X` nào và vì sao đáp án là `3`?
2. **Dự đoán output:** `build_suffix([3,5,2])` cho list nào? Vì sao list dài `n+1`?
3. **Điền TODO:** tự viết hàm `exercise_suffix` dựng mảng suffix.
4. **Tự viết:** dựa trên ý nghĩa `dp(i,M)`, hoàn thiện hàm `exercise_stone_game` không sao chép lời giải.
5. **Biến thể:** sửa hàm để tìm cả **`X` tối ưu ở lượt đầu**, nếu hòa thì lấy `X` nhỏ nhất. Giải thích vì sao đầu ra biến thể không trùng signature LeetCode.

In [16]:
# BÀI TẬP — các hàm TODO chỉ được định nghĩa, KHÔNG chạy tự động.
def exercise_suffix(piles):
    # TODO: tạo n+1 số 0; duyệt ngược; return bảng suffix.
    pass

def exercise_stone_game(piles):
    # TODO: tính suffix, viết dfs(i,M) có @cache, return dfs(0,1).
    pass

def exercise_first_move(piles):
    # TODO: trả về (số đá tối đa, X tối ưu ở lượt đầu).
    # Nếu có nhiều X tối ưu, chọn X nhỏ nhất.
    pass

# Sau khi tự làm, có thể bỏ chú thích để tự kiểm tra:
# assert exercise_suffix([3,5,2]) == [10,7,2,0]
# assert exercise_stone_game([2,7,9,4,4]) == 10
# assert exercise_first_move([2,7,9,4,4]) == (10,1)

## 24. Gợi ý theo 3 mức — chưa xem đáp án nhé

| Bài | Gợi ý 1 — ý tưởng | Gợi ý 2 — cú pháp | Gợi ý 3 — gần code |
|---|---|---|---|
| 1 | Ban đầu M=1 | X=1 hoặc 2 | Lấy cả 2 ⇒ 3 viên |
| 2 | Tổng từ cuối về | `range(n-1,-1,-1)` | `s[i] = piles[i] + s[i+1]` |
| 3 | Mảng có `n+1` ô | `[0]*(n+1)` | Gọi quy tắc bài 2 trong loop |
| 4 | Ta lưu đáp án người sắp đi | `@cache` | `max(suffix[i]-dfs(i+X,max(M,X)))` |
| 5 | So các `X` mở đầu | Lặp X tăng dần | Chỉ cập nhật nếu kết quả **lớn hơn** hiện tại |

## 25. Đáp án tham khảo — chỉ đọc sau khi tự làm

**Bài 1:** `X=1,2`; chọn `X=2` Alice lấy cả `1+2=3`.  
**Bài 2:** `[10,7,2,0]`, ô cuối bằng 0 đại diện đoạn rỗng.  
**Bài 3–5:** code tham khảo bên dưới. Bài 5 trả tuple chỉ để luyện tập, **không** phải dạng submit của LeetCode #1140.

In [17]:
def answer_suffix(piles):
    s=[0]*(len(piles)+1)
    for i in range(len(piles)-1,-1,-1):
        s[i]=piles[i]+s[i+1]
    return s

def answer_stone_game(piles):
    s=answer_suffix(piles)
    n=len(piles)
    @cache
    def dp(i,M):
        if 2*M >= n-i: return s[i]
        return max(s[i]-dp(i+X,max(M,X)) for X in range(1,2*M+1))
    return dp(0,1)

def answer_first_move(piles):
    n=len(piles)
    s=answer_suffix(piles)
    @cache
    def dp(i,M):
        if 2*M >= n-i: return s[i]
        return max(s[i]-dp(i+X,max(M,X)) for X in range(1,2*M+1))
    best=-1
    first=None
    for X in range(1,min(2,n)+1):
        gained=s[0]-dp(X,max(1,X))
        if gained > best:  # dấu > giữ X nhỏ nhất khi hòa
            best,first=gained,X
    return best,first

assert answer_suffix([3,5,2]) == [10,7,2,0]
assert answer_stone_game([2,7,9,4,4]) == 10
assert answer_first_move([2,7,9,4,4]) == (10,1)
print('Đáp án tham khảo: PASS')

Đáp án tham khảo: PASS


## 26. Template tư duy & checklist nộp

```text
1) Xác định trạng thái đủ: (i, M).
2) suffix[i] = tổng đá từ i đến cuối.
3) Nếu 2*M >= n-i -> lấy hết suffix[i].
4) Thử X=1..2*M, chuyển (i+X,max(M,X)).
5) Điểm mình = suffix[i] - điểm tối ưu của đối thủ.
6) Lấy max; cache từng (i,M); đáp án dfs(0,1).
```

- [ ] Đúng số đá trả về (phần Alice), không phải hiệu điểm.
- [ ] `M' = max(M,X)`; không bị reset giữa các lượt.
- [ ] Có base case khi có thể lấy toàn bộ.
- [ ] Không gọi `print` trong code submit.
- [ ] Đã test mẫu `[2,7,9,4,4] -> 10` và `[1,2,3,4,5,100] -> 104`.